# 02 · Modelos clásicos de machine learning
**Proyecto final · Clasificación de tumores cerebrales en resonancia magnética**
Diplomado Liderazgo en la Era de la IA · Módulo 3

Este notebook cubre lo que pide la rúbrica para el Módulo 3:

1. Convierte cada imagen en variables numéricas (escala de grises 64 × 64).
2. Usa **PCA** para entender y reducir los datos, y **K-Means** como apoyo.
3. Compara **cuatro modelos** con **validación cruzada estratificada de 5 pliegues** y las mismas métricas.
4. Evalúa en el set de prueba: matriz de confusión, precisión, sensibilidad y F1.
5. Calcula el **intervalo de confianza** de la exactitud y la **prueba de McNemar** entre modelos.
6. Estima el **costo esperado del error** (falso negativo contra falso positivo).
7. Exporta las tablas para SQL y Power BI.

**Dataset:** Brain Tumor MRI Dataset (Kaggle, Masoud Nickparvar). Estructura esperada:
`datos/Training/{glioma,meningioma,notumor,pituitary}` y `datos/Testing/{...}`.

**Instalación:** `pip install numpy pandas scikit-learn scipy matplotlib seaborn pillow joblib`

## 0. Configuración
Cambia aquí las rutas y los parámetros. Todo lo demás corre sin modificaciones.

In [ ]:
import os
import hashlib
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score, classification_report,
                             confusion_matrix, f1_score, make_scorer, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

# ----- Parámetros (se pueden cambiar con variables de entorno) -----
DATOS = Path(os.environ.get("DATOS", "../datos"))            # carpeta con Training/ y Testing/
SALIDA = Path(os.environ.get("SALIDA", "../resultados_ml"))  # carpeta de resultados
TAM = int(os.environ.get("TAM", 64))                         # lado de la imagen en pixeles
N_COMPONENTES = int(os.environ.get("N_COMPONENTES", 100))    # componentes de PCA
N_PLIEGUES = int(os.environ.get("N_PLIEGUES", 5))            # pliegues de validación cruzada
QUITAR_DUPLICADOS = os.environ.get("QUITAR_DUPLICADOS", "1") == "1"
RAZON_COSTO_FN = float(os.environ.get("RAZON_COSTO_FN", 20)) # un falso negativo cuesta N veces un falso positivo
CSV_RED_NEURONAL = Path(os.environ.get("CSV_RED_NEURONAL", "../resultados_tumores/casos_powerbi.csv"))
SEMILLA = 42

CLASES = ["glioma", "meningioma", "notumor", "pituitary"]
NOMBRES = {"glioma": "Glioma", "meningioma": "Meningioma", "notumor": "Sin tumor", "pituitary": "Tumor hipofisario"}
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SALIDA.mkdir(parents=True, exist_ok=True)
print("Datos:", DATOS.resolve())
print("Resultados:", SALIDA.resolve())

## 1. Carga de imágenes y conversión a variables
Cada imagen se pasa a escala de grises, se redimensiona a 64 × 64 y se aplana en un vector de 4,096 valores entre 0 y 1.
La primera vez tarda uno o dos minutos; después se lee de un archivo de caché.

In [ ]:
def cargar_particion(particion):
    X, y, archivos = [], [], []
    for i, clase in enumerate(CLASES):
        carpeta = DATOS / particion / clase
        if not carpeta.is_dir():
            raise FileNotFoundError(f"No existe {carpeta}. Revisa la variable DATOS.")
        for f in sorted(carpeta.iterdir()):
            if f.suffix.lower() not in EXT:
                continue
            with Image.open(f) as im:
                g = im.convert("L").resize((TAM, TAM), Image.Resampling.BILINEAR)
            X.append(np.asarray(g, dtype=np.float32).ravel() / 255.0)
            y.append(i)
            archivos.append(f.name)
    return np.stack(X), np.array(y), np.array(archivos)

cache = SALIDA / f"imagenes_{TAM}.npz"
if cache.exists():
    d = np.load(cache, allow_pickle=False)
    X_tr, y_tr, a_tr, X_te, y_te, a_te = (d[k] for k in ["X_tr", "y_tr", "a_tr", "X_te", "y_te", "a_te"])
else:
    X_tr, y_tr, a_tr = cargar_particion("Training")
    X_te, y_te, a_te = cargar_particion("Testing")
    np.savez_compressed(cache, X_tr=X_tr, y_tr=y_tr, a_tr=a_tr, X_te=X_te, y_te=y_te, a_te=a_te)

print(f"Entrenamiento: {X_tr.shape} · Prueba: {X_te.shape}")

### 1.1 Duplicados
Si la misma imagen aparece dos veces en entrenamiento, puede caer en pliegues distintos y **inflar** la validación cruzada.
Si aparece en entrenamiento y en prueba, infla la exactitud final. Aquí se detectan copias exactas (después del redimensionado) y se quitan.

In [ ]:
def huella(v):
    return hashlib.md5(np.round(v * 255).astype(np.uint8).tobytes()).hexdigest()

h_tr = np.array([huella(v) for v in X_tr])
h_te = np.array([huella(v) for v in X_te])

_, primeros = np.unique(h_tr, return_index=True)
unicos_tr = np.zeros(len(h_tr), dtype=bool)
unicos_tr[primeros] = True
en_train = np.isin(h_te, h_tr)

resumen_dup = pd.DataFrame({
    "concepto": ["Copias repetidas dentro de entrenamiento", "Imágenes de prueba que también están en entrenamiento"],
    "imagenes": [int((~unicos_tr).sum()), int(en_train.sum())],
    "porcentaje": [round((~unicos_tr).mean() * 100, 2), round(en_train.mean() * 100, 2)],
})
display(resumen_dup)
resumen_dup.to_csv(SALIDA / "duplicados.csv", index=False)

if QUITAR_DUPLICADOS:
    X_tr, y_tr, a_tr = X_tr[unicos_tr], y_tr[unicos_tr], a_tr[unicos_tr]
    X_te, y_te, a_te = X_te[~en_train], y_te[~en_train], a_te[~en_train]
    print(f"Después de quitar duplicados → entrenamiento: {len(y_tr):,} · prueba: {len(y_te):,}")

### 1.2 Distribución por clase

In [ ]:
conteo = pd.DataFrame({
    "clase": [NOMBRES[c] for c in CLASES],
    "entrenamiento": np.bincount(y_tr, minlength=len(CLASES)),
    "prueba": np.bincount(y_te, minlength=len(CLASES)),
})
display(conteo)
ax = conteo.set_index("clase").plot(kind="bar", figsize=(7, 3.5), rot=0)
ax.set_ylabel("Imágenes"); ax.set_xlabel(""); ax.set_title("Imágenes por clase y partición")
plt.tight_layout(); plt.savefig(SALIDA / "01_clases.png", dpi=150); plt.show()

### 1.3 Prueba de hipótesis: ¿el brillo promedio cambia según la clase?
- **H0:** la intensidad media de la imagen tiene la misma distribución en las cuatro clases.
- **Prueba:** Kruskal-Wallis (no supone normalidad).

Si se rechaza H0, hay una advertencia que va en la sección de límites del reporte: un modelo podría acertar por el **brillo o la fuente de la imagen** y no por el tumor.

In [ ]:
brillo = X_tr.mean(axis=1)
grupos = [brillo[y_tr == i] for i in range(len(CLASES))]
H, p_kw = stats.kruskal(*grupos)
print(f"Kruskal-Wallis: H = {H:.1f}, p = {p_kw:.3g}")
print("Se rechaza H0: el brillo difiere entre clases." if p_kw < 0.05 else "No se rechaza H0.")

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(x=[NOMBRES[CLASES[i]] for i in y_tr], y=brillo, order=[NOMBRES[c] for c in CLASES], ax=ax)
ax.set_ylabel("Intensidad media (0 a 1)"); ax.set_xlabel(""); ax.set_title("Brillo promedio por clase")
plt.tight_layout(); plt.savefig(SALIDA / "02_brillo.png", dpi=150); plt.show()

## 2. PCA y K-Means para entender los datos
**PCA** resume los 4,096 pixeles en pocos componentes. La gráfica de varianza acumulada justifica cuántos usar.
**K-Means** con 4 grupos muestra si las clases se separan solas, sin etiquetas.

In [ ]:
n_max = min(300, X_tr.shape[0] - 1, X_tr.shape[1])
Xs = StandardScaler().fit_transform(X_tr)
pca_exp = PCA(n_components=n_max, random_state=SEMILLA).fit(Xs)
acum = np.cumsum(pca_exp.explained_variance_ratio_)
n_usar = min(N_COMPONENTES, n_max)
print(f"Con {n_usar} componentes se conserva {acum[n_usar - 1]:.1%} de la varianza.")

Z = pca_exp.transform(Xs)[:, :n_usar]
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(range(1, n_max + 1), acum)
axs[0].axvline(n_usar, ls="--", color="gray"); axs[0].axhline(acum[n_usar - 1], ls="--", color="gray")
axs[0].set_xlabel("Número de componentes"); axs[0].set_ylabel("Varianza acumulada"); axs[0].set_title("Varianza explicada por PCA")
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=[NOMBRES[CLASES[i]] for i in y_tr], hue_order=[NOMBRES[c] for c in CLASES], s=10, alpha=0.6, ax=axs[1])
axs[1].set_xlabel("Componente 1"); axs[1].set_ylabel("Componente 2"); axs[1].set_title("Clases en los dos primeros componentes")
plt.tight_layout(); plt.savefig(SALIDA / "03_pca.png", dpi=150); plt.show()

km = KMeans(n_clusters=len(CLASES), n_init=10, random_state=SEMILLA).fit(Z)
tabla_km = pd.crosstab(pd.Series([NOMBRES[CLASES[i]] for i in y_tr], name="clase real"), pd.Series(km.labels_, name="grupo K-Means"))
display(tabla_km)
print(f"Índice de Rand ajustado: {adjusted_rand_score(y_tr, km.labels_):.3f} (1 = grupos idénticos a las clases, 0 = al azar)")

## 3. Modelos y validación cruzada
Cada modelo es un **pipeline**: estandarización → PCA → clasificador. Al estar dentro del pipeline, el PCA se ajusta
solo con los datos de entrenamiento de cada pliegue, así que no hay fuga de información.

| Modelo | Idea |
|---|---|
| Regresión logística | Línea base lineal, fácil de explicar |
| SVM con kernel RBF | Fronteras no lineales; suele ir bien con PCA |
| Random Forest | Conjunto de árboles; robusto al ruido |
| K-NN | Clasifica por las imágenes más parecidas |

Métricas: exactitud, precisión, sensibilidad (recall) y F1, todas en promedio macro para que las cuatro clases pesen igual.

In [ ]:
def pipeline(modelo):
    return Pipeline([
        ("escala", StandardScaler()),
        ("pca", PCA(n_components=n_usar, random_state=SEMILLA)),
        ("modelo", modelo),
    ])

MODELOS = {
    "Regresión logística": pipeline(LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")),
    "SVM (RBF)": pipeline(SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", probability=True, random_state=SEMILLA)),
    "Random Forest": pipeline(RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEMILLA)),
    "K-NN": pipeline(KNeighborsClassifier(n_neighbors=5, weights="distance")),
}
METRICAS = {"exactitud": "accuracy", "precision": make_scorer(precision_score, average="macro", zero_division=0),
            "sensibilidad": "recall_macro", "f1": "f1_macro"}
cv = StratifiedKFold(n_splits=N_PLIEGUES, shuffle=True, random_state=SEMILLA)

filas, pliegues = [], []
for nombre, modelo in MODELOS.items():
    r = cross_validate(modelo, X_tr, y_tr, cv=cv, scoring=METRICAS, n_jobs=-1)
    fila = {"modelo": nombre}
    for m in METRICAS:
        fila[f"{m}_media"] = r[f"test_{m}"].mean()
        fila[f"{m}_std"] = r[f"test_{m}"].std()
        for k, v in enumerate(r[f"test_{m}"], 1):
            pliegues.append({"modelo": nombre, "metrica": m, "pliegue": k, "valor": v})
    fila["segundos_entrenamiento"] = r["fit_time"].mean()
    filas.append(fila)
    print(f"{nombre:22s} F1 = {fila['f1_media']:.4f} ± {fila['f1_std']:.4f}")

cv_tabla = pd.DataFrame(filas).sort_values("f1_media", ascending=False).reset_index(drop=True)
cv_tabla.to_csv(SALIDA / "validacion_cruzada.csv", index=False)
pd.DataFrame(pliegues).to_csv(SALIDA / "validacion_cruzada_pliegues.csv", index=False)
display(cv_tabla.round(4))

fig, ax = plt.subplots(figsize=(8, 3.8))
sns.boxplot(data=pd.DataFrame(pliegues).query("metrica == 'f1'"), x="modelo", y="valor", order=cv_tabla.modelo, ax=ax)
ax.set_ylabel("F1 macro por pliegue"); ax.set_xlabel(""); ax.set_title(f"Validación cruzada de {N_PLIEGUES} pliegues")
plt.tight_layout(); plt.savefig(SALIDA / "04_validacion_cruzada.png", dpi=150); plt.show()

MEJOR = cv_tabla.modelo[0]
print("Mejor modelo en validación cruzada:", MEJOR)

## 4. Evaluación final en el set de prueba
Cada modelo se entrena con todo el entrenamiento y se evalúa **una sola vez** en prueba.
Además de las métricas por clase, se calcula la **sensibilidad para detectar tumor** (cualquier tipo contra sin tumor),
que es la métrica de negocio: un tumor que el modelo marca como "sin tumor" es el error más caro.

In [ ]:
def wilson(aciertos, n, z=1.96):
    """Intervalo de confianza de Wilson al 95 % para una proporción."""
    p = aciertos / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - margen, centro + margen

SIN_TUMOR = CLASES.index("notumor")
pred, proba, filas = {}, {}, []
for nombre, modelo in MODELOS.items():
    modelo.fit(X_tr, y_tr)
    pred[nombre] = modelo.predict(X_te)
    proba[nombre] = modelo.predict_proba(X_te)
    p = pred[nombre]
    li, ls = wilson((p == y_te).sum(), len(y_te))
    real_tumor, pred_tumor = y_te != SIN_TUMOR, p != SIN_TUMOR
    fn = int((real_tumor & ~pred_tumor).sum())   # tumor que el modelo llamó "sin tumor"
    fp = int((~real_tumor & pred_tumor).sum())   # sano que el modelo llamó "tumor"
    filas.append({
        "modelo": nombre,
        "exactitud": accuracy_score(y_te, p), "ic95_inferior": li, "ic95_superior": ls,
        "precision": precision_score(y_te, p, average="macro", zero_division=0),
        "sensibilidad": recall_score(y_te, p, average="macro", zero_division=0),
        "f1": f1_score(y_te, p, average="macro", zero_division=0),
        "sensibilidad_tumor": 1 - fn / max(1, real_tumor.sum()),
        "falsos_negativos": fn, "falsos_positivos": fp,
        "costo_error": fn * RAZON_COSTO_FN + fp,
    })

prueba_tabla = pd.DataFrame(filas).sort_values("f1", ascending=False).reset_index(drop=True)
prueba_tabla.to_csv(SALIDA / "resultados_prueba.csv", index=False)
display(prueba_tabla.round(4))

fig, axs = plt.subplots(1, len(MODELOS), figsize=(4.2 * len(MODELOS), 4))
for ax, nombre in zip(np.atleast_1d(axs), MODELOS):
    cm = confusion_matrix(y_te, pred[nombre], labels=range(len(CLASES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=[NOMBRES[c] for c in CLASES], yticklabels=[NOMBRES[c] for c in CLASES])
    ax.set_title(nombre); ax.set_xlabel("Predicción"); ax.set_ylabel("Clase real")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(SALIDA / "05_matrices_confusion.png", dpi=150); plt.show()

print(f"Reporte por clase del mejor modelo ({MEJOR}):")
print(classification_report(y_te, pred[MEJOR], labels=range(len(CLASES)), target_names=[NOMBRES[c] for c in CLASES], zero_division=0))

### 4.1 Costo esperado del error
Unidad de costo: **1 = una lectura prioritaria innecesaria (falso positivo)**.
Un falso negativo cuesta `RAZON_COSTO_FN` veces más (por defecto 20; es un supuesto que conviene validar con un radiólogo).

$$C_{error} = FN \times C_{FN} + FP \times C_{FP}$$

El modelo con mejor exactitud no siempre es el de menor costo. Esa comparación es la que pide la rúbrica: *métricas justificadas por el costo de cada error*.

In [ ]:
orden = prueba_tabla.sort_values("costo_error")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(orden.modelo, orden.falsos_negativos * RAZON_COSTO_FN, label=f"Falsos negativos × {RAZON_COSTO_FN:g}")
ax.barh(orden.modelo, orden.falsos_positivos, left=orden.falsos_negativos * RAZON_COSTO_FN, label="Falsos positivos × 1")
ax.invert_yaxis(); ax.set_xlabel("Costo esperado del error (unidades)"); ax.legend()
ax.set_title("Costo del error por modelo en el set de prueba")
plt.tight_layout(); plt.savefig(SALIDA / "06_costo_error.png", dpi=150); plt.show()
print("Modelo de menor costo:", orden.modelo.iloc[0])

## 5. Prueba de McNemar: ¿la diferencia entre modelos es real?
Dos modelos evaluados sobre las mismas imágenes se comparan con la prueba de McNemar. Solo cuentan los casos en que **uno acierta y el otro falla**:

- **b:** el modelo A acierta y el B falla.
- **c:** el modelo A falla y el B acierta.
- **H0:** los dos modelos se equivocan con la misma frecuencia.

Con menos de 25 casos discordantes se usa la versión exacta (binomial); con más, la ji cuadrada con corrección de continuidad.

In [ ]:
def mcnemar(acierta_a, acierta_b):
    b = int((acierta_a & ~acierta_b).sum())
    c = int((~acierta_a & acierta_b).sum())
    if b + c == 0:
        return b, c, 1.0, "sin casos discordantes"
    if b + c < 25:
        return b, c, float(stats.binomtest(min(b, c), b + c, 0.5).pvalue), "exacta (binomial)"
    est = (abs(b - c) - 1) ** 2 / (b + c)
    return b, c, float(stats.chi2.sf(est, 1)), "ji cuadrada con corrección"

filas = []
nombres = list(prueba_tabla.modelo)
for i in range(len(nombres)):
    for j in range(i + 1, len(nombres)):
        A, B = nombres[i], nombres[j]
        b, c, p, metodo = mcnemar(pred[A] == y_te, pred[B] == y_te)
        filas.append({"modelo_a": A, "modelo_b": B, "a_acierta_b_falla": b, "a_falla_b_acierta": c,
                      "p_valor": p, "metodo": metodo, "diferencia_significativa": p < 0.05})
mcnemar_tabla = pd.DataFrame(filas)
mcnemar_tabla.to_csv(SALIDA / "mcnemar.csv", index=False)
display(mcnemar_tabla.round(4))

### 5.1 Mejor modelo clásico contra la red neuronal en Keras
Esta celda lee el archivo `casos_powerbi.csv` que genera `tumores_cerebrales.py` y compara, imagen por imagen,
la red neuronal contra el mejor modelo clásico. Si el archivo todavía no existe, la celda avisa y el notebook sigue.

In [ ]:
if CSV_RED_NEURONAL.exists():
    red = pd.read_csv(CSV_RED_NEURONAL)
    clasico = pd.DataFrame({"archivo": a_te, "clase_real": [CLASES[i] for i in y_te],
                            "acierta_clasico": pred[MEJOR] == y_te})
    red["archivo_base"] = red.archivo.map(lambda s: Path(str(s)).stem)
    clasico["archivo_base"] = clasico.archivo.map(lambda s: Path(str(s)).stem)
    junto = clasico.merge(red[["archivo_base", "clase_real", "correcto"]], on=["archivo_base", "clase_real"], how="inner")
    acierta_red = junto.correcto.astype(str).str.lower().isin(["true", "1", "verdadero"]).to_numpy()
    b, c, p, metodo = mcnemar(junto.acierta_clasico.to_numpy(), acierta_red)
    print(f"Imágenes comparadas: {len(junto):,}")
    print(f"Exactitud {MEJOR}: {junto.acierta_clasico.mean():.4f} · Exactitud red neuronal: {acierta_red.mean():.4f}")
    print(f"{MEJOR} acierta y la red falla: {b} · {MEJOR} falla y la red acierta: {c}")
    print(f"McNemar ({metodo}): p = {p:.4g} →", "diferencia significativa" if p < 0.05 else "sin diferencia significativa")
    pd.DataFrame([{"modelo_a": MEJOR, "modelo_b": "Red neuronal (Keras)", "a_acierta_b_falla": b,
                   "a_falla_b_acierta": c, "p_valor": p, "metodo": metodo, "imagenes": len(junto)}]).to_csv(SALIDA / "mcnemar_vs_red.csv", index=False)
else:
    print(f"No se encontró {CSV_RED_NEURONAL}. Ejecuta primero tumores_cerebrales.py y vuelve a correr esta celda.")

## 6. Exportación para SQL y Power BI
- `predicciones_ml.csv`: una fila por imagen y por modelo. Es la base de la tabla de hechos `fact_prediccion`.
- `dim_modelo.csv` y `dim_clase.csv`: dimensiones del modelo estrella.
- `mejor_modelo.joblib`: el mejor pipeline, listo para usarse en una aplicación o como herramienta del agente.

In [ ]:
partes = []
for nombre in MODELOS:
    p, pr = pred[nombre], proba[nombre]
    partes.append(pd.DataFrame({
        "archivo": a_te,
        "modelo": nombre,
        "clase_real": [CLASES[i] for i in y_te],
        "prediccion": [CLASES[i] for i in p],
        "confianza": pr.max(axis=1).astype(float).round(4),
        "correcto": (p == y_te).astype(int),
        "falso_negativo_tumor": ((y_te != SIN_TUMOR) & (p == SIN_TUMOR)).astype(int),
        "falso_positivo_tumor": ((y_te == SIN_TUMOR) & (p != SIN_TUMOR)).astype(int),
    }))
predicciones = pd.concat(partes, ignore_index=True)
predicciones.to_csv(SALIDA / "predicciones_ml.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({"modelo": list(MODELOS), "tipo": "Machine learning clásico",
              "es_mejor_cv": [m == MEJOR for m in MODELOS]}).to_csv(SALIDA / "dim_modelo.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"clase": CLASES, "nombre": [NOMBRES[c] for c in CLASES],
              "es_tumor": [int(c != "notumor") for c in CLASES]}).to_csv(SALIDA / "dim_clase.csv", index=False, encoding="utf-8-sig")
joblib.dump(MODELOS[MEJOR], SALIDA / "mejor_modelo.joblib")

print(f"Filas en predicciones_ml.csv: {len(predicciones):,}")
print("Archivos generados:")
for f in sorted(SALIDA.iterdir()):
    print("  ", f.name)

## 7. Resumen para el reporte
Esta celda arma el párrafo de resultados con las cifras de esta corrida.

In [ ]:
m = prueba_tabla.set_index("modelo").loc[MEJOR]
cvm = cv_tabla.set_index("modelo").loc[MEJOR]
print(f"""Se compararon {len(MODELOS)} modelos con validación cruzada estratificada de {N_PLIEGUES} pliegues
sobre {len(y_tr):,} imágenes de entrenamiento ({n_usar} componentes de PCA).

Mejor modelo: {MEJOR}
  · F1 macro en validación cruzada: {cvm.f1_media:.3f} ± {cvm.f1_std:.3f}
  · Exactitud en prueba ({len(y_te):,} imágenes): {m.exactitud:.3f} (IC 95 %: {m.ic95_inferior:.3f} a {m.ic95_superior:.3f})
  · F1 macro en prueba: {m.f1:.3f}
  · Sensibilidad para detectar tumor: {m.sensibilidad_tumor:.3f}
  · Falsos negativos: {int(m.falsos_negativos)} · Falsos positivos: {int(m.falsos_positivos)}

Kruskal-Wallis sobre el brillo por clase: p = {p_kw:.3g}.""")